# Variable Neighborhood Search (VNS)

## Overview

VNS is a metaheuristic that systematically explores different neighborhood structures to escape local optima and find better solutions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

print("VNS Tutorial")

## VNS Core Components

In [ ]:
class VNS:
    def __init__(self, problem, max_iter=100):
        self.problem = problem
        self.max_iter = max_iter
        self.best_solution = None
        self.best_fitness = float('inf')
        
    def shake(self, solution, k):
        """Shaking operation with k neighborhoods"""
        new_solution = solution.copy()
        
        if k == 1:
            # Small perturbation
            idx = random.randint(0, len(solution) - 1)
            new_solution[idx] += np.random.normal(0, 0.1)
        elif k == 2:
            # Medium perturbation
            idx1, idx2 = random.sample(range(len(solution)), 2)
            new_solution[idx1], new_solution[idx2] = new_solution[idx2], new_solution[idx1]
        else:
            # Large perturbation
            new_solution = np.random.uniform(-5, 5, len(solution))
        
        return np.clip(new_solution, -5, 5)
    
    def local_search(self, solution):
        """Simple local search"""
        current = solution.copy()
        current_fitness = self.problem.evaluate(current)
        
        improved = True
        while improved:
            improved = False
            for i in range(len(current)):
                for delta in [-0.1, 0.1]:
                    neighbor = current.copy()
                    neighbor[i] += delta
                    neighbor = np.clip(neighbor, -5, 5)
                    
                    neighbor_fitness = self.problem.evaluate(neighbor)
                    if neighbor_fitness < current_fitness:
                        current = neighbor
                        current_fitness = neighbor_fitness
                        improved = True
        
        return current, current_fitness
    
    def neighborhood_change(self, current, best, k):
        """Change neighborhood structure"""
        current_fitness = self.problem.evaluate(current)
        best_fitness = self.problem.evaluate(best)
        
        if current_fitness < best_fitness:
            return current, 1  # Reset to first neighborhood
        else:
            return best, k + 1  # Move to next neighborhood
    
    def optimize(self, initial_solution=None):
        """VNS main algorithm"""
        # Initialize
        if initial_solution is None:
            self.best_solution = np.random.uniform(-5, 5, self.problem.dimension)
        else:
            self.best_solution = initial_solution.copy()
        
        self.best_fitness = self.problem.evaluate(self.best_solution)
        
        print(f"Starting VNS with initial fitness: {self.best_fitness:.4f}")
        
        for iteration in range(self.max_iter):
            k = 1
            while k <= 3:  # Use 3 neighborhood structures
                # Shaking
                shaken_solution = self.shake(self.best_solution, k)
                
                # Local search
                local_solution, local_fitness = self.local_search(shaken_solution)
                
                # Neighborhood change
                self.best_solution, k = self.neighborhood_change(
                    local_solution, self.best_solution, k
                )
                
                self.best_fitness = self.problem.evaluate(self.best_solution)
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Best fitness = {self.best_fitness:.4f}")
        
        return self.best_solution, self.best_fitness

# Define test problem
class SphereProblem:
    def __init__(self, dimension=5):
        self.dimension = dimension
    
    def evaluate(self, x):
        return np.sum(x**2)

# Test VNS
problem = SphereProblem(dimension=5)
vns = VNS(problem, max_iter=100)
best_sol, best_fit = vns.optimize()

print(f"\nVNS complete!")
print(f"Best solution: {best_sol}")
print(f"Best fitness: {best_fit:.4f}")

## VNS with Bias System

In [ ]:
class VNSWithBias(VNS):
    def __init__(self, problem, bias_system=None, max_iter=100):
        super().__init__(problem, max_iter)
        self.bias_system = bias_system
        self.history = []
    
    def biased_shake(self, solution, k):
        """Shaking with bias guidance"""
        if self.bias_system and np.random.rand() < 0.3:
            # Apply bias direction
            bias_direction = np.random.randn(len(solution)) * 0.5
            solution = solution + bias_direction
        
        return self.shake(solution, k)
    
    def optimize(self, initial_solution=None):
        if initial_solution is None:
            self.best_solution = np.random.uniform(-5, 5, self.problem.dimension)
        else:
            self.best_solution = initial_solution.copy()
        
        self.best_fitness = self.problem.evaluate(self.best_solution)
        
        print(f"Starting VNS with Bias, initial fitness: {self.best_fitness:.4f}")
        
        for iteration in range(self.max_iter):
            k = 1
            while k <= 3:
                # Biased shaking
                shaken_solution = self.biased_shake(self.best_solution, k)
                
                # Local search
                local_solution, local_fitness = self.local_search(shaken_solution)
                
                # Neighborhood change
                self.best_solution, k = self.neighborhood_change(
                    local_solution, self.best_solution, k
                )
                
                self.best_fitness = self.problem.evaluate(self.best_solution)
            
            self.history.append(self.best_fitness)
            
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Best fitness = {self.best_fitness:.4f}")
        
        return self.best_solution, self.best_fitness

# Test VNS with bias
print("\nTesting VNS with Bias...")
vns_bias = VNSWithBias(problem, max_iter=100)
best_sol_bias, best_fit_bias = vns_bias.optimize()

print(f"\nVNS with Bias complete!")
print(f"Best fitness: {best_fit_bias:.4f}")

## Comparison Visualization

In [ ]:
# Compare VNS with and without bias
plt.figure(figsize=(10, 4))

# Multiple runs for comparison
n_runs = 10
results_no_bias = []
results_with_bias = []

for run in range(n_runs):
    vns_nb = VNS(problem, max_iter=50)
    _, fit_nb = vns_nb.optimize()
    results_no_bias.append(fit_nb)
    
    vns_b = VNSWithBias(problem, max_iter=50)
    _, fit_b = vns_b.optimize()
    results_with_bias.append(fit_b)

# Plot results
plt.subplot(1, 2, 1)
plt.boxplot([results_no_bias, results_with_bias], 
           labels=['VNS', 'VNS + Bias'])
plt.ylabel('Final Fitness')
plt.title('Performance Comparison')
plt.grid(True, alpha=0.3)

# Convergence example
plt.subplot(1, 2, 2)
vns_example = VNSWithBias(problem, max_iter=50)
vns_example.optimize()
plt.plot(vns_example.history, 'b-', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Best Fitness')
plt.title('Convergence Curve')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage performance:")
print(f"VNS without bias: {np.mean(results_no_bias):.4f}")
print(f"VNS with bias: {np.mean(results_with_bias):.4f}")

## Summary

VNS key features:
- Systematic neighborhood exploration
- Shaking and local search phases
- Escapes local optima effectively
- Bias system enhances convergence

## Next Tutorial

Next: [Algorithm Nesting and Hybrid Strategies](07_Algorithm_Nesting_and_Hybrid_Strategies.ipynb)